In [37]:
from train_gpt2 import GPT2 
import torch
import tiktoken

In [38]:
my_model = GPT2.from_pretrained()
my_model.eval()

device = "mps" if torch.backends.mps.is_available() else "cpu"
my_model.to(device)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 726.62it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x TransformerBlock(
        (attn): Attention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [39]:
enc = tiktoken.get_encoding("gpt2")

assert enc.decode(enc.encode("Hello world")) == "Hello world" 

In [49]:
phrase = "What is the meaning of life?" 
tokens = enc.encode(phrase) 

input_ids = torch.tensor(tokens, device=device)
# input_ids

In [ ]:
torch.manual_seed(42)
torch.mps.manual_seed(42)
max_length = 30 

current = input_ids.unsqueeze(0).expand(5, -1)

with torch.no_grad():
    while current.shape[1] < max_length:
        # print(current.shape)
        output = my_model(current)
        next_token_logits = output[:, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)
        topk_probs, topk_indices = torch.topk(next_token_probs, k=50) 
        next_token = torch.multinomial(topk_probs, num_samples=1)
        xcol = torch.gather(topk_indices, -1, next_token)
        current = torch.cat((current, xcol.squeeze(0)), dim=1)

In [51]:
for i in range(current.shape[0]):
    print(f"Generated sequence {i}: {enc.decode(current[i].tolist())}")

Generated sequence 0: What is the meaning of life? If it is the intention of life? If it is is the intention the intent the intent the the the the the
Generated sequence 1: What is the meaning of life? It was, then, then, then, then, then, then, second second second second second final final final
Generated sequence 2: What is the meaning of life? The meaning of life is the meaning of life

The meaning of life is the Meaning of life

The
Generated sequence 3: What is the meaning of life? It is the mean, meaning, meaning, meaning, meaning, meaning, meaning, meaning, meaning, meaning,
Generated sequence 4: What is the meaning of life? meaning meaning meaning meaning meaning meaning meaning meaning meaning Meaning Meaning Meaning Meaning Meaning Meaning Meaning Meaning Meaning Meaning Meaning Meaning Meaning Meaning
